# CatBoost pipeline for Dota death-probability prediction

Этот ноутбук — аналог пайплайна с логистической регрессией, но для **CatBoost**.

Почему CatBoost здесь уместен:
- он умеет **нативно работать с категориальными признаками**;
- для категорий не нужен one-hot encoding;
- на табличных данных это часто сильный baseline и хороший следующий шаг после логрегрессии.

Внутри ноутбука:
- загрузка и проверка данных;
- аккуратный split по `match_id`, чтобы не было leakage между снапшотами одного матча;
- обучение CatBoost с early stopping;
- перебор нескольких конфигураций;
- выбор порога;
- метрики, ROC/PR-кривые, матрица ошибок, важности признаков;
- сохранение модели.

## Почему CatBoost, а не one-hot пайплайн

CatBoost сам строит преобразования для категориальных признаков.  
Это особенно полезно для hero id из Dota: вместо ручного OHE модель может использовать категориальные фичи напрямую.

Практический смысл:
- меньше ручной подготовки;
- обычно лучше качество на табличных данных;
- проще поддерживать prod inference.

In [11]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from catboost import CatBoostClassifier, Pool

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
)

import json
from pandas.api.types import is_numeric_dtype

In [12]:
# =========================
# Configuration
# =========================


DATA_PATH = Path("../full_dataset.csv")
HERO_NAMES_PATH = Path("../heroNames.json") # надо для преобразования имен героев в их id

TARGET_COL = "dead_in_1"
MATCH_ID_COL = "match_id" 
TIME_COL = "game_time"

HERO_COLS = ["hero_id", "enemy_1_name", "enemy_2_name", "enemy_3_name", "enemy_4_name", "enemy_5_name"]

RANDOM_STATE = 120
TEST_SIZE = 0.15
VAL_SIZE = 0.15

ARTIFACT_DIR = Path("artifacts/boosting")
ARTIFACT_DIR.mkdir(exist_ok=True)

# Грубая сетка для подбора гиперпараметров.
# Это не "идеальный" search, а хороший и быстрый baseline.
CANDIDATE_PARAMS = [
    {"iterations": 3000, "depth": 6, "learning_rate": 0.03, "l2_leaf_reg": 8.0},
    {"iterations": 4000, "depth": 6, "learning_rate": 0.02, "l2_leaf_reg": 10.0},
    {"iterations": 3000, "depth": 8, "learning_rate": 0.03, "l2_leaf_reg": 12.0},
    {"iterations": 5000, "depth": 4, "learning_rate": 0.05, "l2_leaf_reg": 6.0},
]

THRESHOLD_GRID = np.linspace(0.05, 0.95, 181)

## Загрузка данных

Базовые проверки:
- таргет есть;
- `match_id` есть;
- hero-колонки есть;
- NaN / inf аккуратно нормализуем.

CatBoost умеет работать с пропусками в числовых признаках, поэтому отдельный sklearn-imputer здесь не обязателен.  
Но категориальные колонки лучше привести к явному виду, чтобы не было сюрпризов.

In [13]:
def load_data(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Файл не найден: {path.resolve()}")

    suffix = path.suffix.lower()
    if suffix == ".csv":
        df = pd.read_csv(path)
    elif suffix in {".parquet", ".pq"}:
        df = pd.read_parquet(path)
    elif suffix == ".feather":
        df = pd.read_feather(path)
    else:
        raise ValueError(f"Неподдерживаемый формат: {suffix}")

    return df


df = load_data(DATA_PATH)
print(df.shape)
df.head()

(714600, 106)


,id,match_id,game_time,is_day,is_radiant,radiant_score,dire_score,hero_id,level,kills,...,item_sphere,item_manta,item_blade_mail,item_aeon_disk,item_pipe,dead_in_1,dead_in_5,dead_in_10,dead_in_15,dead_in_20
0,1,8766917279,43,1,1,0,0,86,1,0,...,0,0,0,0,0,0,0,0,0,0
1,2,8766917279,65,1,1,0,0,129,1,0,...,0,0,0,0,0,0,0,0,0,1
2,3,8766917279,68,1,1,0,0,129,1,0,...,0,0,0,0,0,0,0,0,1,1
3,4,8766917279,77,1,1,0,0,129,1,0,...,0,0,0,0,0,0,0,1,1,1
4,5,8766917279,80,1,1,0,0,129,1,0,...,0,0,0,0,0,0,1,1,1,1


In [14]:
missing_cols = [c for c in [TARGET_COL, MATCH_ID_COL] + HERO_COLS if c not in df.columns]

print("Missing columns:", missing_cols)

assert TARGET_COL in df.columns, f"Нет target-колонки: {TARGET_COL}"
assert MATCH_ID_COL in df.columns, f"Нет колонки для группировки матчей: {MATCH_ID_COL}"

print("Target distribution:")
print(df[TARGET_COL].value_counts(dropna=False))
print()
print("Target rate:")
print(df[TARGET_COL].mean())

dup_count = df.duplicated().sum()
print(f"Duplicates: {dup_count}")

na_share = (df.isna().mean().sort_values(ascending=False).head(15))
print("\nTop missingness:")
print(na_share)

Missing columns: []
Target distribution:
dead_in_1
0    592889
1    121711
Name: count, dtype: int64

Target rate:
0.17032045899804085
Duplicates: 0

Top missingness:
enemy_1_name     0.000067
enemy_5_name     0.000034
game_time        0.000000
is_day           0.000000
is_radiant       0.000000
radiant_score    0.000000
dire_score       0.000000
hero_id          0.000000
id               0.000000
match_id         0.000000
deaths           0.000000
assists          0.000000
last_hits        0.000000
denies           0.000000
gold             0.000000
dtype: float64


In [15]:
def transform_name_to_id(df: pd.DataFrame, hero_cols: list[str]) -> pd.DataFrame:
    with open(HERO_NAMES_PATH, "r") as f:
        data = json.load(f)
    
    for col in hero_cols:
        if is_numeric_dtype(df[col]):
            continue  # уже в виде id, пропускаем
        df[col] = df[col].apply(lambda x: data.get(x, pd.NA))
        df[col] = df[col].astype("Int64")  # используем nullable integer dtype для сохранения NA

    return df

ENEMIES_NUMBER = 5
def sort_enemies(df: pd.DataFrame) -> pd.DataFrame:
    enemy_names_cols = [f"enemy_{i}_name" for i in range(1, ENEMIES_NUMBER + 1)]
    enemy_x_cols = [f"enemy_{i}_last_seen_x" for i in range(1, ENEMIES_NUMBER + 1)]
    enemy_y_cols = [f"enemy_{i}_last_seen_y" for i in range(1, ENEMIES_NUMBER + 1)]
    enemy_square_cols = [f"enemy_{i}_last_seen_sqare" for i in range(1, ENEMIES_NUMBER + 1)]
    enemy_distance_cols = [f"enemy_{i}_last_seen_distance" for i in range(1, ENEMIES_NUMBER + 1)]
    enemy_time_cols = [f"enemy_{i}_last_seen_time" for i in range(1, ENEMIES_NUMBER + 1)]

    enemy_names = df[enemy_names_cols].to_numpy()
    enemy_x = df[enemy_x_cols].to_numpy()
    enemy_y = df[enemy_y_cols].to_numpy()
    enemy_square = df[enemy_square_cols].to_numpy()
    enemy_distance = df[enemy_distance_cols].to_numpy()
    enemy_time = df[enemy_time_cols].to_numpy()

    order = np.argsort(enemy_distance, axis=1)

    sorted_enemy_names = np.take_along_axis(enemy_names, order, axis=1)
    sorted_enemy_x = np.take_along_axis(enemy_x, order, axis=1)
    sorted_enemy_y = np.take_along_axis(enemy_y, order, axis=1)
    sorted_enemy_square = np.take_along_axis(enemy_square, order, axis=1)
    sorted_enemy_distance = np.take_along_axis(enemy_distance, order, axis=1)
    sorted_enemy_time = np.take_along_axis(enemy_time, order, axis=1)

    df[enemy_names_cols] = sorted_enemy_names
    df[enemy_x_cols] = sorted_enemy_x
    df[enemy_y_cols] = sorted_enemy_y
    df[enemy_square_cols] = sorted_enemy_square
    df[enemy_distance_cols] = sorted_enemy_distance
    df[enemy_time_cols] = sorted_enemy_time

    return df


df = df.pipe(transform_name_to_id, HERO_COLS).pipe(sort_enemies).dropna()
df.shape

(714528, 106)

## Split по матчам

Это ключевой момент.

Если у тебя много снапшотов из одного и того же матча, то обычный random split почти наверняка даст leakage:
один и тот же матч частично попадёт и в train, и в val/test.

Поэтому split делаем по `match_id`:  
все строки одного матча должны жить только в одной части датасета.

In [20]:
def split_by_groups(
    df: pd.DataFrame,
    group_col: str,
    test_size: float = TEST_SIZE,
    val_size: float = VAL_SIZE,
    random_state: int = 42,
):
    if not (0 < test_size < 1 and 0 < val_size < 1):
        raise ValueError("test_size и val_size должны быть в (0, 1)")

    if test_size + val_size >= 1:
        raise ValueError("test_size + val_size должно быть < 1")

    # copy() нужен потому что новые pandas/numpy
    # могут возвращать read-only ndarray
    groups = (
        df[group_col]
        .drop_duplicates()
        .to_numpy()
        .copy()
    )

    rng = np.random.default_rng(random_state)
    rng.shuffle(groups)

    n_groups = len(groups)

    n_test = int(round(n_groups * test_size))
    n_val = int(round(n_groups * val_size))

    test_groups = set(groups[:n_test])
    val_groups = set(groups[n_test:n_test + n_val])
    train_groups = set(groups[n_test + n_val:])

    train_df = df[df[group_col].isin(train_groups)].copy()
    val_df = df[df[group_col].isin(val_groups)].copy()
    test_df = df[df[group_col].isin(test_groups)].copy()

    return train_df, val_df, test_df


train_df, val_df, test_df = split_by_groups(
    df,
    group_col=MATCH_ID_COL,
    test_size=TEST_SIZE,
    val_size=VAL_SIZE,
    random_state=RANDOM_STATE,
)

print("train:", train_df.shape)
print("val:  ", val_df.shape)
print("test: ", test_df.shape)

print()

for name, part in [
    ("train", train_df),
    ("val", val_df),
    ("test", test_df),
]:
    print(
        f"{name:<5} target mean:",
        round(part[TARGET_COL].mean(), 6),
    )

train: (440147, 106)
val:   (184038, 106)
test:  (90343, 106)

train target mean: 0.170925
val   target mean: 0.16928
test  target mean: 0.169587


## Подготовка матриц признаков и `Pool`

CatBoost любит `Pool`-объекты: в них явно передаются:
- признаки;
- таргет;
- список категориальных колонок.

Для нас это удобно, потому что:
- не нужен `OneHotEncoder`;
- не нужен `StandardScaler`;
- CatBoost сам выберет, как работать с числовыми и категориальными признаками.

In [25]:
COLS_TO_DROP = [TARGET_COL, MATCH_ID_COL, "id", "slot_1_id", "slot_2_id", "slot_3_id", "slot_4_id", "slot_5_id", "dead_in_1", "dead_in_5", "dead_in_10", "dead_in_15", "dead_in_20"]

HERO_COLS_EXISTING = [c for c in HERO_COLS if c in df.columns]

feature_cols = [c for c in df.columns if c not in COLS_TO_DROP]
categorical_cols = HERO_COLS_EXISTING
numeric_cols = [c for c in feature_cols if c not in categorical_cols]


print("Numeric cols:", len(numeric_cols))

X_train = train_df[feature_cols].copy()
y_train = train_df[TARGET_COL].copy()

X_val = val_df[feature_cols].copy()
y_val = val_df[TARGET_COL].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df[TARGET_COL].copy()

for X in (X_train, X_val, X_test):
    for col in HERO_COLS:
        X[col] = X[col].astype("category").fillna("__MISSING__")

train_pool = Pool(X_train, y_train, cat_features=HERO_COLS)
val_pool = Pool(X_val, y_val, cat_features=HERO_COLS)
test_pool = Pool(X_test, y_test, cat_features=HERO_COLS)

print("n_features:", len(feature_cols))
print("categorical features:", HERO_COLS)

Numeric cols: 88
n_features: 94
categorical features: ['hero_id', 'enemy_1_name', 'enemy_2_name', 'enemy_3_name', 'enemy_4_name', 'enemy_5_name']


## Функция обучения

Здесь параметры подобраны как разумный стартовый baseline:

- `loss_function='Logloss'` — бинарная классификация;
- `eval_metric='AUC'` — удобно следить за ранжированием на validation;
- `depth` — контролирует сложность деревьев;
- `learning_rate` + `iterations` — классический trade-off: чем меньше learning rate, тем больше деревьев нужно;
- `l2_leaf_reg` — регуляризация листьев, чтобы модель не переобучалась;
- `od_type='Iter'`, `od_wait=...` — early stopping;
- `use_best_model=True` — оставить только лучшую итерацию по validation;
- `thread_count=-1` — использовать все ядра CPU.

Параметры ниже не считаются единственно верными; это хороший старт, от которого можно двигаться дальше.

In [36]:
def make_model(
    *,
    iterations: int,
    depth: int,
    learning_rate: float,
    l2_leaf_reg: float,
    random_state: int = RANDOM_STATE,
    class_weights=None,
) -> CatBoostClassifier:
    return CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="AUC",
        iterations=iterations,
        depth=depth,
        learning_rate=learning_rate,
        l2_leaf_reg=l2_leaf_reg,
        random_seed=random_state,
        od_type="Iter",
        od_wait=200,
        use_best_model=True,
        verbose=200,
        thread_count=-1,
        class_weights=class_weights,
    )

## Метрики

Считаем те же метрики, что и для логрегрессии:
- logloss
- ROC-AUC
- PR-AUC
- Brier score
- accuracy
- precision
- recall
- F1

Плюс отдельно будем подбирать порог по validation, чтобы не застревать в `0.5` как в единственно правильном значении.

In [37]:
def evaluate_probabilities(y_true, y_proba, threshold=0.5):
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)
    y_pred = (y_proba >= threshold).astype(int)

    return {
        "logloss": log_loss(y_true, y_proba, labels=[0, 1]),
        "roc_auc": roc_auc_score(y_true, y_proba),
        "pr_auc": average_precision_score(y_true, y_proba),
        "brier": brier_score_loss(y_true, y_proba),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }


def choose_best_threshold(y_true, y_proba, threshold_grid=THRESHOLD_GRID, metric="f1"):
    rows = []
    for t in threshold_grid:
        y_pred = (y_proba >= t).astype(int)
        rows.append({
            "threshold": float(t),
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
        })
    scores = pd.DataFrame(rows)
    best_idx = scores[metric].idxmax()
    return scores, scores.loc[best_idx, "threshold"], scores.loc[best_idx]

## Подбор гиперпараметров

Смысл такой же, как у перебора `C` в логрегрессии:
мы не пытаемся найти “идеальную” модель с первого раза, а сначала ищем разумный диапазон сложности.

Для CatBoost основные ручки:
- `depth` — глубина деревьев;
- `learning_rate` — шаг обучения;
- `iterations` — сколько деревьев строить;
- `l2_leaf_reg` — регуляризация.

Обычно полезно начинать с небольшой сетки и только потом расширять поиск.

In [38]:
search_results = []

for params in CANDIDATE_PARAMS:
    model = make_model(**params)
    model.fit(train_pool, eval_set=val_pool)

    val_proba = model.predict_proba(val_pool)[:, 1]
    metrics = evaluate_probabilities(y_val, val_proba, threshold=0.5)

    search_results.append({
        **params,
        **metrics,
        "best_iteration": model.get_best_iteration(),
    })

search_df = (
    pd.DataFrame(search_results)
    .sort_values(["logloss", "roc_auc"], ascending=[True, False])
    .reset_index(drop=True)
)

search_df

0:	test: 0.7586036	best: 0.7586036 (0)	total: 234ms	remaining: 11m 40s
200:	test: 0.7962053	best: 0.7962053 (200)	total: 35s	remaining: 8m 7s
400:	test: 0.8002600	best: 0.8002600 (400)	total: 1m 15s	remaining: 8m 12s
600:	test: 0.8029595	best: 0.8029708 (599)	total: 1m 56s	remaining: 7m 46s
800:	test: 0.8041898	best: 0.8041898 (800)	total: 2m 39s	remaining: 7m 17s
1000:	test: 0.8051073	best: 0.8051073 (1000)	total: 3m 22s	remaining: 6m 44s
1200:	test: 0.8058167	best: 0.8058167 (1200)	total: 4m 5s	remaining: 6m 7s
1400:	test: 0.8063200	best: 0.8063220 (1398)	total: 4m 49s	remaining: 5m 29s
1600:	test: 0.8068044	best: 0.8068044 (1600)	total: 5m 36s	remaining: 4m 53s
1800:	test: 0.8070930	best: 0.8071144 (1797)	total: 6m 23s	remaining: 4m 15s
2000:	test: 0.8074306	best: 0.8074385 (1989)	total: 7m 8s	remaining: 3m 34s
2200:	test: 0.8076994	best: 0.8077118 (2197)	total: 7m 52s	remaining: 2m 51s
2400:	test: 0.8078892	best: 0.8078921 (2397)	total: 8m 37s	remaining: 2m 9s
2600:	test: 0.8081073

KeyboardInterrupt: 

## Выбор лучшей конфигурации и обучение финальной модели

Берём лучшую строку из search-таблицы по `logloss`, а при равенстве — по `roc_auc`.

Дальше:
- обучаем модель на train;
- смотрим validation;
- выбираем порог;
- один раз честно считаем test.

In [ ]:
best_row = search_df.iloc[0].to_dict()
best_params = {
    "iterations": int(best_row["iterations"]),
    "depth": int(best_row["depth"]),
    "learning_rate": float(best_row["learning_rate"]),
    "l2_leaf_reg": float(best_row["l2_leaf_reg"]),
}

best_params

In [ ]:
final_model = make_model(**best_params)
final_model.fit(train_pool, eval_set=val_pool)

val_proba = final_model.predict_proba(val_pool)[:, 1]
test_proba = final_model.predict_proba(test_pool)[:, 1]

threshold_scores, best_threshold, best_threshold_row = choose_best_threshold(y_val, val_proba, metric="f1")

print("best_threshold:", best_threshold)
print(best_threshold_row)

val_metrics = evaluate_probabilities(y_val, val_proba, threshold=best_threshold)
test_metrics = evaluate_probabilities(y_test, test_proba, threshold=best_threshold)

metrics_df = pd.DataFrame([
    {"split": "validation", **val_metrics},
    {"split": "test", **test_metrics},
])

metrics_df

## Кривые качества

### ROC-кривая
Показывает trade-off между TPR и FPR для разных порогов.

### PR-кривая
Показывает trade-off между precision и recall.  
Для несбалансированных классов обычно важнее ROC-кривой.

### История обучения
Если всё хорошо, validation-метрика сначала улучшается, а потом перестаёт улучшаться — и early stopping останавливает обучение.

In [ ]:
def plot_roc_pr_curves(y_true, y_proba, title_prefix="Test"):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    fpr, tpr, _ = roc_curve(y_true, y_proba)
    axes[0].plot(fpr, tpr, label=f"ROC-AUC = {roc_auc_score(y_true, y_proba):.4f}")
    axes[0].plot([0, 1], [0, 1], linestyle="--", label="random baseline")
    axes[0].set_xlabel("FPR")
    axes[0].set_ylabel("TPR")
    axes[0].set_title(f"{title_prefix}: ROC curve")
    axes[0].legend()

    precision, recall, _ = precision_recall_curve(y_true, y_proba)
    ap = average_precision_score(y_true, y_proba)
    axes[1].plot(recall, precision, label=f"PR-AUC = {ap:.4f}")
    axes[1].set_xlabel("Recall")
    axes[1].set_ylabel("Precision")
    axes[1].set_title(f"{title_prefix}: PR curve")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


plot_roc_pr_curves(y_test, test_proba, title_prefix="Test")

In [ ]:
def plot_training_history(model: CatBoostClassifier):
    evals = model.get_evals_result()
    print("evals_result keys:", list(evals.keys()))

    train_key = next((k for k in evals.keys() if "learn" in k or "train" in k), None)
    val_key = next((k for k in evals.keys() if k != train_key), None)

    if train_key is None or val_key is None:
        print("Не удалось найти обе части eval history.")
        return

    common_metrics = sorted(set(evals[train_key].keys()) & set(evals[val_key].keys()))
    if not common_metrics:
        print("Нет общих метрик для plot.")
        return

    metric = "Logloss" if "Logloss" in common_metrics else common_metrics[0]

    plt.figure(figsize=(10, 5))
    plt.plot(evals[train_key][metric], label=f"train {metric}")
    plt.plot(evals[val_key][metric], label=f"val {metric}")
    plt.xlabel("Iteration")
    plt.ylabel(metric)
    plt.title(f"Training history: {metric}")
    plt.legend()
    plt.tight_layout()
    plt.show()


plot_training_history(final_model)

In [ ]:
# Матрица ошибок на test при выбранном пороге
test_pred = (test_proba >= best_threshold).astype(int)
cm = confusion_matrix(y_test, test_pred)

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(cm, interpolation="nearest")
ax.set_title("Confusion matrix (test)")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["0", "1"])
ax.set_yticklabels(["0", "1"])

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center")

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# Важность признаков
importances = final_model.get_feature_importance(train_pool)
fi = (
    pd.DataFrame({
        "feature": feature_cols,
        "importance": importances,
    })
    .sort_values("importance", ascending=False)
    .head(25)
    .sort_values("importance", ascending=True)
)

plt.figure(figsize=(10, 8))
plt.barh(fi["feature"], fi["importance"])
plt.title("Top feature importances")
plt.tight_layout()
plt.show()

fi

## Сохранение модели

CatBoost удобно сохранять в `.cbm` файл.  
Это хороший формат для prod inference: модель можно загрузить без повторного обучения.

In [ ]:
s = TARGET_COL[-1] + "s"

MODEL_PATH = ARTIFACT_DIR / Path(f"catboost_dota_model_{s}.cbm")
final_model.save_model(str(MODEL_PATH))
print(f"Saved to {MODEL_PATH.resolve()}")

## Как читать результат

Если модель:
- даёт ROC-AUC заметно выше 0.5;
- показывает разумный PR-AUC;
- не переобучается слишком сильно;
- имеет стабильную validation curve;

то это хороший следующий шаг после логрегрессии.

Если же CatBoost почти не улучшает baseline, это не обязательно ошибка.  
Иногда это означает, что:
- признаки уже извлекли почти весь сигнал;
- нужно лучшее feature engineering;
- нужен другой target;
- или задача действительно ограничена качеством данных.